In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.7 Solving Ordinary Differential Equations

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.7",
    title="Solving Ordinary Differential Equations",
    blurb="The engine under every simulation in this course: Euler to "
    "Runge–Kutta, the order of a method, the stiffness that wrecks "
    "explicit schemes, and why an adaptive solver spends its effort "
    "where the solution actually moves.",
    difficulty="advanced",
    estimate="100–130 min",
)

## Notebook overview

Every simulation in this course has secretly been an exercise in solving
ordinary differential equations: the projectile of
[§1.1](../01-elementary-mechanics/projectile-drag.ipynb), the chaotic pendulum
of [§1.3](../01-elementary-mechanics/double-pendulum.ipynb), the orbits of
[§1.4](../01-elementary-mechanics/kepler-orbits.ipynb), the phase flow of
[§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb): all of them
handed their
equations of motion to `scipy.integrate.solve_ivp` and trusted the trajectory
that came back. This notebook is the general theory behind that trust. It builds
the integrators from the ground up, asks what makes one *order* better than
another, and then confronts the two failure modes that the order alone never
predicts: **stiffness**, which can make a perfectly accurate explicit method
explode, and the slow **energy drift** of a high-order method over long times,
the phenomenon [§1.6](../01-elementary-mechanics/integrators.ipynb) first
showed in one case and which we now explain in general.

We start from Euler's one-line tangent step and work up to Runge–Kutta, measure
convergence orders exactly as [§0.3](quadrature-differentiation.ipynb) measured
them for quadrature, then
meet the new idea that separates a textbook solver from a production one: that
**stability** is a constraint distinct from accuracy, and that taming a stiff
problem means going *implicit*, which, satisfyingly, turns each step into a
root-find of the kind built in [§0.2](root-finding.ipynb). We finish with the
adaptive step-size control that `solve_ivp` actually uses, and a hard-won
lesson from [§1.6](../01-elementary-mechanics/integrators.ipynb):
for a Hamiltonian system over long times, the *geometry* of a method matters more
than its order.

There are no animations here: these are studies of error, step size, and energy,
which a still plot reads more honestly than a moving one would.

> **How to read the checks.** Each exercise ends with a `validate` call against
> an independent fact: an exact solution $e^{-t}$, a predicted convergence order,
> a stability threshold. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*, not a verdict.

> **Scope.** A working review, not a course in numerical ODEs. The standard
> reference is Hairer, Nørsett & Wanner, *Solving Ordinary Differential
> Equations* {cite}`hairer`; see also Press et al., *Numerical Recipes*, ch. 17
> {cite}`numrecipes`. The long-time geometry is the subject of
> [§1.6](../01-elementary-mechanics/integrators.ipynb).

## Theory in brief

### The initial-value problem

Everything here solves the **initial-value problem**: given a rate law and a
starting state, find the trajectory. Written for a (possibly vector) state
$\mathbf y$,

```{math}
:label: eq-ivp
\mathbf y'(t) = \mathbf f(t, \mathbf y), \qquad \mathbf y(t_0) = \mathbf y_0 .
```

A higher-order ODE is not a separate case: it reduces to this one by stacking
derivatives into the state, the trick the course has used since
[§1.1](../01-elementary-mechanics/projectile-drag.ipynb). The
oscillator $y''=-y$, for instance, becomes the first-order system for
$(x, v) = (y, y')$,

$$
\begin{pmatrix} x \\ v \end{pmatrix}' =
\begin{pmatrix} v \\ -x \end{pmatrix},
$$

so a single solver for {eq}`eq-ivp` handles everything.

### Euler's method and the order of a method

The simplest integrator follows the tangent: knowing the slope $\mathbf f$ at the
current point, step along it by $h$,

```{math}
:label: eq-euler-step
\mathbf y_{n+1} = \mathbf y_n + h\,\mathbf f(t_n, \mathbf y_n).
```

It is first order, and what "order" means here is exactly what it meant for
quadrature in [§0.3](quadrature-differentiation.ipynb): a method has **order
$p$** if its global error over a
fixed interval scales as $O(h^p)$,

```{math}
:label: eq-odeorder
\lVert \mathbf y_N - \mathbf y(t_{\text{end}})\rVert \approx C\,h^{\,p}
  \approx C'\,n^{-p},
```

so $p$ is again minus the slope of a log–log error-vs-$n$ plot. (The local error
made in one step is one power higher, $O(h^{p+1})$; accumulating $\sim 1/h$ steps
costs that one power.)

### Runge–Kutta: sampling the slope

Euler is crude because it trusts the slope at the *start* of the step for the
whole step. **Runge–Kutta** methods do better by sampling $\mathbf f$ at
intermediate points and combining the samples so the leading error terms cancel.
The midpoint rule (RK2) takes a half-step to probe the slope at the middle; the
classic **RK4** blends four such samples,

```{math}
:label: eq-rk-step
\mathbf y_{n+1} = \mathbf y_n + \tfrac{h}{6}\,(\mathbf k_1 + 2\mathbf k_2 + 2\mathbf k_3 + \mathbf k_4),
```

with $\mathbf k_1=\mathbf f(t_n,\mathbf y_n)$, $\mathbf k_2=\mathbf f(t_n+\tfrac
h2,\mathbf y_n+\tfrac h2\mathbf k_1)$, $\mathbf k_3=\mathbf f(t_n+\tfrac h2,
\mathbf y_n+\tfrac h2\mathbf k_2)$, $\mathbf k_4=\mathbf f(t_n+h,\mathbf y_n+h
\mathbf k_3)$. RK4 is fourth order and the general-purpose default for non-stiff
problems. (The coefficients are fixed by matching the Taylor expansion of the
exact flow through $O(h^4)$; Hairer, Nørsett & Wanner, *Solving Ordinary
Differential Equations*, develop the order conditions in full.)

### Stability is not accuracy: stiffness

Order controls how fast the error falls as $h\to0$; it says nothing about whether
a *given* $h$ blows up. For the decaying test equation $y'=-k y$ (true solution
$e^{-kt}\to0$), one Euler step multiplies $y$ by $1-kh$, so the numerical
solution stays bounded only when

```{math}
:label: eq-stiffness
|1 - kh| \le 1 \quad\Longleftrightarrow\quad h \le \frac{2}{k}.
```

A **stiff** system (one with widely separated timescales, i.e. a large $k$)
therefore forces an explicit method to take absurdly small steps for *stability*,
long after accuracy stopped demanding it. This is the key new idea of the
notebook, and order cannot fix it.

### Implicit methods and adaptivity

The cure is to evaluate the slope at the *end* of the step instead. **Backward
Euler**,

```{math}
:label: eq-beuler
\mathbf y_{n+1} = \mathbf y_n + h\,\mathbf f(t_{n+1}, \mathbf y_{n+1}),
```

has $\mathbf y_{n+1}$ on both sides, so each step is an equation to be *solved*
(a root-find, [§0.2](root-finding.ipynb), or, for a nonlinear $\mathbf f$, a Newton iteration
per step); the payoff is unconditional stability at any $h$. Production stiff
solvers (BDF, Radau) are implicit for exactly this reason. Separately, real
solvers do not use a fixed $h$ at all: they **estimate the local error** with an
embedded pair of methods (RK45 / Dormand–Prince) and shrink or grow $h$ to hold
it under a tolerance, spending effort where the solution actually moves. That is
what `solve_ivp` does. And, as
[§1.6](../01-elementary-mechanics/integrators.ipynb) showed and we revisit at
the end,
none of this captures long-time *geometry*: a non-symplectic method, however
high its order, slowly drifts a Hamiltonian system's energy.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from ecp import validate


def euler(f, y0, t):
    """Explicit (forward) Euler on a uniform grid (eq-euler-step of the theory section).

    First-order: simple but its error and stability are the cautionary baseline.

    Parameters
    ----------
    f : callable
        Right-hand side ``f(t, y)``.
    y0 : array_like
        Initial state.
    t : numpy.ndarray
        Uniform time grid.

    Returns
    -------
    numpy.ndarray
        The solution at each grid time.
    """
    y = np.empty((len(t),) + np.shape(y0))
    y[0] = y0
    for n in range(len(t) - 1):
        h = t[n + 1] - t[n]
        y[n + 1] = y[n] + h * np.asarray(f(t[n], y[n]))
    return y


def rk2(f, y0, t):
    """Explicit midpoint Runge–Kutta (second order).

    Parameters
    ----------
    f : callable
        Right-hand side.
    y0 : array_like
        Initial state.
    t : numpy.ndarray
        Uniform time grid.

    Returns
    -------
    numpy.ndarray
        The solution history.
    """
    y = np.empty((len(t),) + np.shape(y0))
    y[0] = y0
    for n in range(len(t) - 1):
        h = t[n + 1] - t[n]
        k1 = np.asarray(f(t[n], y[n]))
        k2 = np.asarray(f(t[n] + h / 2, y[n] + h / 2 * k1))
        y[n + 1] = y[n] + h * k2
    return y


def rk4(f, y0, t):
    """Classic four-stage Runge–Kutta (fourth order, eq-rk-step of the theory section).

    The standard explicit workhorse: four slope evaluations per step for fourth-order accuracy.

    Parameters
    ----------
    f : callable
        Right-hand side.
    y0 : array_like
        Initial state.
    t : numpy.ndarray
        Uniform time grid.

    Returns
    -------
    numpy.ndarray
        The solution history.
    """
    y = np.empty((len(t),) + np.shape(y0))
    y[0] = y0
    for n in range(len(t) - 1):
        h = t[n + 1] - t[n]
        k1 = np.asarray(f(t[n], y[n]))
        k2 = np.asarray(f(t[n] + h / 2, y[n] + h / 2 * k1))
        k3 = np.asarray(f(t[n] + h / 2, y[n] + h / 2 * k2))
        k4 = np.asarray(f(t[n] + h, y[n] + h * k3))
        y[n + 1] = y[n] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return y


def fit_order(ns, errs):
    """Empirical convergence order p from an error-vs-step series.

    Minus the slope of log(error) vs log(n), fitted above the round-off floor.

    Parameters
    ----------
    ns : array_like
        Step counts (or inverse step sizes).
    errs : array_like
        Corresponding errors.

    Returns
    -------
    float
        The estimated order $p$.
    """
    ns, errs = np.asarray(ns, float), np.asarray(errs, float)
    m = errs > 1e-13
    return -np.polyfit(np.log(ns[m]), np.log(errs[m]), 1)[0]

## Exercise 1 — Euler's method, and reducing a higher-order ODE

We meet the integrator on the problem with the simplest possible exact answer.
The scalar IVP $y' = -y$, $y(0) = 1$ has the solution $y(t) = e^{-t}$, so any
error is laid bare against a known curve. Euler's tangent step {eq}`eq-euler-step`
turns the rate law into a recurrence; it should crawl toward $e^{-t}$ as the step
shrinks. The same machine also handles second-order ODEs once they are written as
first-order systems, {eq}`eq-ivp`: the oscillator $y''=-y$ with $y(0)=1,\,
y'(0)=0$ becomes $(x,v)'=(v,-x)$, whose exact solution is $(\cos t, -\sin t)$.

1. Integrate $y'=-y$ on $[0,5]$ with the `euler` helper at a fine step and
   confirm it tracks $e^{-t}$.
2. Integrate the reduced oscillator system on $[0,2\pi]$ and confirm it matches
   $(\cos t, -\sin t)$.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(y_decay, np.exp(-t1), "Euler converges to the exact e^{-t}", rtol=1e-2)
validate.close(
    sol_osc, exact_osc, "the reduced (x,v) system matches (cos t, -sin t)", atol=1e-2
)
# (Euler is first order and slightly *inflates* the oscillator's amplitude over a
# period (the energy-drift effect this notebook explains in Ex. 7), so the
# tolerance is set for a first-order method, not machine precision.)

## Exercise 2 — The order of a method

A method's worth is set by its order, and we measure it the same way
[§0.3](quadrature-differentiation.ipynb) measured quadrature orders: run the
method at a range of step counts and read the slope of the error.

1. On the IVP $y'=-y$, $y(0)=1$ over $[0,1]$ (exact value $y(1)=e^{-1}$), compute
   the global error at $t=1$ for Euler, the midpoint RK2, and RK4 over a
   geometric sweep of step counts.
2. Plot the errors log–log ({numref}`fig-ode-order`).
3. Fit the slopes with the `fit_order` helper (`numpy.polyfit` in log space):
   by {eq}`eq-odeorder` they should be $-1$, $-2$, and $-4$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(p_euler, 1.0, "Euler is first order", atol=0.15)
validate.close(p_rk2, 2.0, "midpoint RK2 is second order", atol=0.15)
validate.close(p_rk4, 4.0, "RK4 is fourth order", atol=0.2)

## Exercise 3 — Runge–Kutta on a nonlinear problem

The order test used a linear equation; the point of Runge–Kutta is that the same
fourth-order accuracy carries over to nonlinear problems, where no closed form is
at hand. Take the explicit nonlinear IVP $y' = t - y^2$, $y(0)=1$ on $[0,2]$.

1. Lacking an elementary solution to check against, build a trustworthy
   **reference** by running the `rk4` helper at a very fine step.
2. Confirm that RK4 at a coarse step (64 steps) already agrees with it: the
   slope-sampling of {eq}`eq-rk-step` proving its worth.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    y_coarse,
    y_ref,
    "RK4 matches the fine reference on the nonlinear IVP y'=t-y²",
    rtol=1e-5,
)

## Exercise 4 — Stiffness: when explicit methods explode

Here is the centrepiece, and the idea that order alone never warns us about. The
IVP $y' = -50\,y$, $y(0)=1$ on $[0,1]$ has the gently decaying exact solution
$e^{-50t}$, yet explicit Euler blows up on it unless the step respects the
**stability** bound $h \le 2/k = 2/50 = 0.04$ of {eq}`eq-stiffness`. With $n=10$
steps ($h=0.1 > 0.04$) the numerical solution oscillates and explodes to $\sim
10^6$; with $n=30$ ($h\approx0.033 < 0.04$) it stays bounded, its amplitude decaying even as its sign flips each step
({numref}`fig-ode-stiff`). The instability has nothing to do with accuracy (the
true solution could hardly be calmer) and everything to do with the step.

1. Integrate with the `euler` helper at $n=10$ ($h=0.1$) and $n=30$
   ($h\approx0.033$), on either side of the stability bound.
2. Confirm the unstable run diverges while the stable one stays bounded.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    abs(y_unstable[-1]) > 1e3 and abs(y_stable[-1]) < 1.0,
    "explicit Euler is unstable when h exceeds 2/k, despite the true solution decaying",
    f"|y(1)|: h=0.1 → {abs(y_unstable[-1]):.1e}, h=0.033 → {abs(y_stable[-1]):.1e}",
)

## Exercise 5 — Implicit methods and unconditional stability

If evaluating the slope at the *start* of the step is what makes Euler unstable,
evaluating it at the *end* is the fix. Backward Euler {eq}`eq-beuler` puts
$y_{n+1}$ on both sides; for the linear stiff IVP $y'=-50\,y$ the per-step
equation $y_{n+1} = y_n - 50\,h\,y_{n+1}$ solves algebraically to $y_{n+1} =
y_n/(1+50h)$, whose multiplier $1/(1+50h)$ has magnitude below one for *every*
$h>0$: unconditional stability. (For a nonlinear $\mathbf f$ the per-step
equation would instead need a root-find, exactly the machinery of
[§0.2](root-finding.ipynb).)

1. Implement the linear backward-Euler step (the `backward_euler_linear` helper).
2. Run it at $h=0.1$, the very step where explicit Euler exploded, and watch it
   decay.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    abs(y_be[-1]) < 1.0,
    "backward Euler is stable at a step (h=0.1) where explicit Euler blew up",
    f"backward Euler |y(1)| = {abs(y_be[-1]):.2e}",
)

## Exercise 6 — Adaptive step-size control

A production solver does not commit to one step size; it watches its own local
error and adjusts $h$ on the fly, taking small steps through fast transitions and
large ones across slow arcs. To see this, integrate the **Van der Pol
oscillator** (a self-sustaining nonlinear system with sharp relaxation kicks)
at $\mu=5$: as a first-order system, $x'=v$ and $v'=\mu(1-x^2)v - x$, with
$x(0)=2,\ v(0)=0$ on $[0,20]$.

1. Solve it with `scipy.integrate.solve_ivp` (the adaptive RK45 /
   Dormand–Prince pair).
2. Look at the steps it *chose* (`numpy.diff` of `sol.t`): clustered tightly
   where the trajectory snaps, spread out where it glides
   ({numref}`fig-ode-adaptive`). A static plot of the accepted step points shows
   this far more honestly than a real-time march would.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    dt.max() / dt.min() > 10,
    "the adaptive solver varies its step size by more than 10× over the trajectory",
    f"max/min step ratio = {dt.max() / dt.min():.0f}×",
)

## Exercise 7 — Order isn't everything: energy drift (student exercise)

A closing caution, and the general statement of what
[§1.6](../01-elementary-mechanics/integrators.ipynb) showed in one
case: a high order buys short-term accuracy, but it does not buy long-term
*geometric* fidelity. Integrate the Hamiltonian IVP $x''=-x$ (the unit harmonic
oscillator), $x(0)=1,\ x'(0)=0$ as the system $(x,v)'=(v,-x)$ on the long interval
$[0,200]$, and track the energy $E=\tfrac12(x^2+v^2)$, which is exactly conserved
by the true motion.

1. Integrate with `scipy.integrate.solve_ivp` (RK45) and follow $E(t)$.
2. Show its energy error *grows* with time — a secular drift — despite the high
   order.
3. Contrast with the symplectic **velocity-Verlet** scheme of
   [§1.6](../01-elementary-mechanics/integrators.ipynb), whose energy
   error oscillates in a *bounded* band forever ({numref}`fig-ode-drift`).

A ✗ points at the energy bookkeeping, not at any drawing.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    growth_rk > 1.5 and growth_vv < 1.5,
    "the non-symplectic RK45's energy error GROWS with time (secular drift) while "
    "the symplectic Verlet's stays bounded",
    f"error-growth ratio: RK45 {growth_rk:.2f} (>1.5), Verlet {growth_vv:.2f} (<1.5)",
)

## Exercise 8 — Choosing a solver in practice (synthesis)

The decisions reduce to three questions, each answered by something seen above. Is
the problem **non-stiff**? An explicit Runge–Kutta (`solve_ivp`'s default RK45)
is fast and accurate. Is it **stiff** (widely separated timescales, an explicit
method forced to crawl for stability)? Reach for an *implicit* solver,
`method="BDF"` or `"Radau"`, which inherits backward Euler's unconditional
stability. Is it a **Hamiltonian system integrated over very long times**? Prefer
a symplectic scheme (velocity-Verlet,
[§1.6](../01-elementary-mechanics/integrators.ipynb)) for bounded energy.

1. To close the loop, re-solve the stiff IVP $y'=-50\,y$, $y(0)=1$ on $[0,1]$
   (the one that wrecked explicit Euler) with the implicit
   `solve_ivp(..., method="Radau")`.
2. Confirm it matches $e^{-50t}$.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    y_radau,
    np.exp(-50 * t_chk),
    "an implicit (Radau) solver handles the stiff problem accurately",
    rtol=1e-4,
    atol=1e-8,
)

## Notebook summary

- Euler's method (first order, converging to $e^{-t}$), the measured order of a scheme, and
  Runge–Kutta (RK4 fourth order) on a nonlinear problem.
- **Stiffness** making explicit methods explode, implicit methods and their unconditional
  stability, adaptive step-size control, the energy-drift caveat, and how to choose a solver
  (`scipy.integrate.solve_ivp`) in practice.

## Outlook

- **Multistep methods** (Adams–Bashforth/Moulton, BDF) reuse past samples instead
  of recomputing intermediate stages, trading memory and start-up cost for
  cheaper steps.
- **Boundary-value problems** replace "given the start, march forward" with
  conditions at both ends; *shooting* turns them back into an IVP plus a root-find
  ([§0.2](root-finding.ipynb)): the route to the hydrogen radial equation in
  [§6.17](../06-quantum-mechanics/hydrogen-atom.ipynb).
- **Symplectic integrators in depth**
  ([§1.6](../01-elementary-mechanics/integrators.ipynb), and molecular dynamics
  in Volume VI) are the tool whenever long-time energy behaviour matters.
- **Event detection** stops or branches the integration at a condition (a turning
  point, an impact) as the scattering integrator of
  [§2.5](../02-classical-mechanics/scattering.ipynb) already does.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()